In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score 

import mlflow
from responsibleai import RAIInsights
from raiwidgets import ResponsibleAIDashboard
import time

# ==========================================
# 1. Load Data
# ==========================================
print(">>> [Step 1] Loading Data...")
try:
    train_df = pd.read_csv('Sensitive 0.1 codinate from nose train.csv')
    test_df = pd.read_csv('Sensitive 0.1 codinate from nose test.csv')
except Exception as e:
    print(f"Error: {e}")
    exit()

target_feature = train_df.columns[-1]

# ==========================================
# 2. Data Cleaning
# ==========================================
print(">>> [Step 2] Cleaning Data...")
# Fill missing values to ensure stability
numeric_cols = train_df.select_dtypes(include=[np.number]).columns
train_means = train_df[numeric_cols].mean()
train_df[numeric_cols] = train_df[numeric_cols].fillna(train_means)
test_df[numeric_cols] = test_df[numeric_cols].fillna(train_means)

X_train = train_df.drop(columns=[target_feature])
y_train = train_df[target_feature]
X_test = test_df.drop(columns=[target_feature])
y_test = test_df[target_feature]                

# ==========================================
# 3. Train & Track 
# ==========================================
print(">>> [Step 3] Training & Tracking with MLflow...")

mlflow.set_experiment("Pose_Classification_Tracker")

with mlflow.start_run(run_name="RandomForest_Final"):

    n_estimators = 100
    model = Pipeline([
        ('imputer', SimpleImputer(strategy='mean')),
        ('classifier', RandomForestClassifier(n_estimators=n_estimators, random_state=42))
    ])
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')

    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("f1_score", f1)
  
    
    print(f"Accuracy: {acc:.4f}")

# ==========================================
# 4. Configure RAI Components 
# ==========================================
print(">>> [Step 4] Configuring Dashboard Components...")

rai_insights = RAIInsights(
    model=model,
    train=train_df,
    test=test_df, 
    target_column=target_feature,
    task_type="classification"
)

# Error Analysis
rai_insights.error_analysis.add()

# InterpretML 
rai_insights.explainer.add()

# Data Balance
print(">>> [Step 5] Computing Insights...")
rai_insights.compute()

# ==========================================
# 5. Launch Dashboard
# ==========================================
PORT = 6001
print("="*60)
print("------------------------------------------------")
print(f"1. [Error Analysis] open in browser http://localhost:{PORT}")
print(f"   (这是你的红色树状图)")
print("------------------------------------------------")
print(f"2. [Tracker] keyin in anaconda terminal 'mlflow ui'")
print(f"   Then view in http://localhost:5000 ")
print("="*60)

dashboard = ResponsibleAIDashboard(rai_insights, port=PORT)

while True:
    try:
        time.sleep(1)
    except KeyboardInterrupt:
        break

d:\user\aconda3\envs\rai_toolbox\lib\site-packages\mlflow\utils\autologging_utils\versioning.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


>>> [Step 1] Loading Data...
>>> [Step 2] Cleaning Data...


2026/01/14 18:14:28 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet



>>> [Step 3] Training & Tracking with MLflow...
✅ Tracker 记录成功！Accuracy: 0.5480
>>> [Step 4] Configuring Dashboard Components...
>>> [Step 5] Computing Insights (这可能需要几分钟)...
Causal Effects
Current Status: Generating Causal Effects.
Current Status: Finished generating causal effects.
Time taken: 0.0 min 1.160000000055561e-05 sec
Counterfactual
Time taken: 0.0 min 4.400000001680837e-06 sec
Error Analysis
Current Status: Generating error analysis reports.
Current Status: Finished generating error analysis reports.
Time taken: 0.0 min 2.5057112000000004 sec
Explanations
Current Status: Explaining 195 features
Current Status: Explained 195 features.
Time taken: 0.0 min 6.3860952 sec
🎉 全部完成！请按以下说明截图：
------------------------------------------------
1. [Error Analysis] 浏览器打开: http://localhost:6001
   (这是你的红色树状图)
------------------------------------------------
2. [Tracker] 打开新黑色窗口输入 'mlflow ui'
   然后去 http://localhost:5000 查看表格
ResponsibleAI started at http://localhost:6001
